# 1 · Meet the Beast

Almost every NGSolve session is the **same (at least) three steps**, no matter how hard the
problem: 
* **(1)** set up **geometry and mesh**,
* **(2)** **solve a variational problem** on it,
* **(3)** **visualize** the result.

We will walk that loop **twice** —
once for a linear **Poisson** problem and once for a **nonlinear elasticity** problem —
so you see the shape of everything to come.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "anywidget"], check=True)

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve import solvers
from ngsolve.krylovspace import CGSolver
from ngsolve.webgui import Draw

## 1. Geometry & mesh — the Beast

The Beast is NGSolve's logo **sculpture**, carved from just two primitive shapes — a
**sphere** and a **cylinder** — with **boolean subtraction**. Read the four panels of the
sketch below, left to right:

1. **A solid sphere** — start from a full ball.
2. **Hollow it into a shell** — subtract a smaller, concentric sphere, leaving a thick
   spherical shell.
3. **Line up three bores** — the **green**, **red** and **purple** cylinders, each parallel
   to the $x$-, $y$- or $z$-axis but **offset from the centre** (they do *not* pass through
   it). That off-centre placement is what gives the logo its asymmetric, interlocking look.
4. **The Beast** — subtract those three cylinders; what remains is the sculpture.

![Building the Beast in four steps — a solid sphere, hollowed into a shell, three bores lined up, the finished sculpture](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/beast-construction.jpg)

We build exactly that with **Netgen/OCC** constructive geometry below — the four code
comments are the four panels. Naming a few faces now lets us attach boundary conditions later.

In [ ]:
def beast_sculpture():
    centre = Pnt(50, 50, 50)
    ball  = Sphere(centre, 80)                                   # 1. a solid sphere
    shell = ball - Sphere(centre, 50)                            # 2. hollow it → a thick shell
    shell.faces.name = "shellbnd"
    # 3. three orthogonal bores — the green, red and purple cylinders in the sketch above;
    #    each parallel to an axis but OFFSET from the centre (it does not pass through it).
    bores = [Cylinder(Pnt(-100,    0,    0), X, r=40, h=300),    # green  — parallel to x, offset in y & z
             Cylinder(Pnt( 100, -100,  100), Y, r=40, h=300),    # red    — parallel to y, offset in x & z
             Cylinder(Pnt(   0,  100, -100), Z, r=40, h=300)]    # purple — parallel to z, offset in x & y
    beast = shell
    for bore in bores:                                           # 4. drill them out → the Beast
        bore.faces.name = "borebnd"
        beast = beast - bore
    return beast.Move((-50, -50, -50)).Scale(Pnt(0, 0, 0), 0.05)  # centre + shrink

beast = beast_sculpture()
mesh = Mesh(OCCGeometry(beast).GenerateMesh(maxh=0.5))
mesh.Curve(2)
print(f"the Beast: {mesh.nv} vertices, {mesh.ne} elements, {mesh.GetNE(BND)} surface triangles")
Draw(mesh)

## 2. Heat distribution inside the Beast

Legend says the Beast stores the energy for its fire-breath deep in its body. 

We model that as a **heat source** living inside the shell and ask for the steady temperature —
the **Poisson problem** $-\Delta u = f$ with $u=0$ on the outer surface and $\partial_n u = 0$ on the bore hole surface. 

In NGSolve this is a **weak form** on an `H1` space: find $u$ such that
$$ \underbrace{\int_\Omega \nabla u\cdot\nabla v}_{a(u,\,v)} \;=\; \underbrace{\int_\Omega f\,v}_{f(v)} \qquad\text{for all } v. $$

In [ ]:
u,v = H1(mesh, order=2).TnT()
gfu = Solve( grad(u)*grad(v)*dx == 1*v*dx, u[BND(".*")]==0)

# Clip open and show solution
Draw(gfu, mesh, "temperature", clipping={"function": True, "pnt": (0, 0, 0), "vec": (0, 0, -1)})

## 3. Next example: Stretching a Swiss-cross plate (nonlinear)

To show off **robustness**, the masters grab a stiff specimen — a **Swiss-cross plate**.

**Geometry.** A thin plate $\Omega\subset\mathbb{R}^3$ (width × height × thickness) with a
cross-shaped hole. Its left face $\Gamma_{\mathrm{hold}}$ is **clamped**; its right face
$\Gamma_{\mathrm{pull}}$ carries a horizontal **pulling traction** of strength $g$.

![The Swiss-cross plate — a red slab with a cross-shaped hole, clamped at the left, pulled to the right](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/elasticity-plate.png)

In [ ]:
Wx, Wy, t, bw = 9.0, 6.0, 0.5, 1                   # a landscape plate: width × height × thickness
slab = Box(Pnt(0, 0, 0), Pnt(Wx, Wy, t))
cx, cy = Wx / 2, Wy / 2
vbar = Box(Pnt(cx - bw/2, cy - 2.0, -0.1), Pnt(cx + bw/2, cy + 2.0, t + 0.1))   # vertical cross bar
hbar = Box(Pnt(cx - 2.0, cy - bw/2, -0.1), Pnt(cx + 2.0, cy + bw/2, t + 0.1))   # horizontal cross bar
plate = slab - vbar - hbar                           # the Swiss-cross plate: a cross punched out
plate.faces.Min(X).name = "hold"                     # clamped left edge   Γ_hold
plate.faces.Max(X).name = "pull"                     # pulled right edge   Γ_pull
plate.faces.col = (0.82, 0.0, 0.00)
fmesh = Mesh(OCCGeometry(plate).GenerateMesh(maxh=0.7))
view = dict(euler_angles=[-53.592224997179635,-0.09258685705068664,0.09617530663228589])
Draw(fmesh,**view)

**Kinematics.** The unknown is a **displacement** $\mathbf{u}:\Omega\to\mathbb{R}^3$ moving
each material point $\mathbf{x}\mapsto\mathbf{x}+\mathbf{u}$. Its **deformation gradient** is
$F = I + \nabla\mathbf{u}$, with $J=\det F>0$ and right Cauchy–Green tensor $C=F^{\top}F$.

**Material.** A compressible **Neo-Hookean** hyperelastic stored-energy density
$$ \psi(F) \;=\; \tfrac{\mu}{2}\bigl(\operatorname{tr}C - 3\bigr)\;-\;\mu\,\log J\;+\;\tfrac{\lambda}{2}(\log J)^2,
   \qquad \mu=\frac{E}{2(1+\nu)},\quad \lambda=\frac{E\,\nu}{(1+\nu)(1-2\nu)} . $$

**Variational problem.** The plate settles into the displacement that **minimises the total
potential energy** — stored energy minus the work of the traction — over all admissible
(clamped) displacements:
$$ \mathbf{u}=\arg\min_{\substack{\mathbf{v}\in[H^1(\Omega)]^3\\[1pt]\mathbf{v}=0\ \text{on }\Gamma_{\mathrm{hold}}}}
   \;\Bigl[\;\int_\Omega \psi\bigl(I+\nabla\mathbf{v}\bigr)\,\mathrm{d}x\;-\;\int_{\Gamma_{\mathrm{pull}}} g\,v_1\,\mathrm{d}s\;\Bigr]. $$
Setting its first variation to zero gives a **nonlinear** system; 
* we solve it by **Newton's method** (`solvers.Newton`),
* **ramping** $g$ up from $0$ in small steps
* The same *geometry → weak form → solve → draw* loop
* only the weak form is described through an **energy** now, and the solve **iterates**. 

In [ ]:
E, nu = 200.0, 0.35                                  # Young's modulus, Poisson ratio
mu, lam = E / (2 * (1 + nu)), E * nu / ((1 + nu) * (1 - 2 * nu))
V = VectorH1(fmesh, order=1, dirichlet="hold")
ud = V.TrialFunction()
F = Id(3) + Grad(ud); J = Det(F); C = F.trans * F
psi = 0.5 * mu * (Trace(C) - 3) - mu * log(J) + 0.5 * lam * log(J)**2   # Neo-Hooke

traction = Parameter(0.0)
elastic = BilinearForm(V, symmetric=True)
elastic += Variation(psi * dx)                       # stored elastic energy
elastic += Variation(-traction * ud[0] * ds("pull")) # work of the pulling traction

gfd = GridFunction(V); gfd.vec[:] = 0
morph = GridFunction(V); morph.vec[:] = 0            # frame 0: the original, undeformed flag
nsteps = 12                                          # many small load steps → a slow, smooth morph
for k in range(1, nsteps + 1):                       # ramp the load, keeping continuation
    traction.Set(22.0 * k / nsteps)
    solvers.Newton(elastic, gfd, dampfactor=0.5, printing=False)
    morph.AddMultiDimComponent(gfd.vec)              # one morph frame per load step

We draw it as a **morph**: a multidim field whose frames are the plate at **successive load
steps**, from undeformed to fully pulled. In the webgui it plays on its own (or drag the
**t** slider in the **multidim** menu) to watch it stretch.

In [ ]:
Draw(morph, fmesh, "displacement", deformation=True, interpolate_multidim=True, animate=True, **view)

**Next:** the masters withdraw. To approach the Beast yourself you first conjure it some
**food** — and learn elementary **geometry & meshing** along the way (unit 2).

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("00-setup", "0 · Setting out — installing & running NGSolve")
    _next = ("02-geometry", "2 · Creating Geometry and Meshes")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))